# VS01 단어 정렬 — Colab GPU

**왜 Colab인가**: 이 정렬은 whisper `small`을 41.3시간 오디오에 돌린다. 로컬에는
NVIDIA GPU가 없어(Intel Iris Xe) CPU로 **약 29시간**이 걸린다. T4면 몇 시간이다.

**무엇을 하나**
1. `align_608.py` — whisper 단어 타임스탬프로 264개 부모 wav를 단어 세그먼트로 자른다
2. `refine_segments.py` — 에너지로 경계를 다시 잡고 `task_type`을 채운다 (**필수**)
3. 결과를 드라이브로 회수 → 로컬에서 `prepare_colab_trainset.py`

**준비물**: 드라이브 `MyDrive/`에 `vs01_align_upload.zip` (1.75GiB) 업로드.
스크립트 두 개와 매니페스트가 zip 안에 함께 들어 있다(`VERSION.txt`에 커밋 기록).

> ⚠️ **정렬 결과는 데이터가 된다.** whisper 버전이 바뀌면 세그먼트 경계가 달라져
> 기존 배치와 성격이 어긋난다. 그래서 로컬 정렬과 같은 `openai-whisper==20250625`로
> 핀을 박는다. 모델도 `small` 고정 — 이전 5개 배치가 전부 small이었다.

## 셀 1 — GPU 확인 (T4가 떠야 한다)

In [ ]:
!nvidia-smi

## 셀 2 — 설치

`openai-whisper`만 있으면 된다(ffmpeg·numpy는 Colab에 이미 있다).
**버전을 핀으로 박는다** — 위 경고 참고.

In [ ]:
WHISPER_PIN = "20250625"   # 로컬 정렬과 동일. 바꾸면 이전 배치와 경계가 달라진다

!pip install -q "openai-whisper=={WHISPER_PIN}"

import importlib.metadata as _m, torch, subprocess
got = _m.version("openai-whisper")
print("whisper:", got, "| torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("ffmpeg :", subprocess.run(["ffmpeg","-version"], capture_output=True, text=True)
      .stdout.splitlines()[0])
assert got == WHISPER_PIN, f"핀이 안 먹었다: {got}"
assert torch.cuda.is_available(), "GPU가 없다 — 런타임 유형을 T4로 바꿔라"

## 셀 3 — 드라이브 마운트 + 업로드 풀기

오디오는 `/content`(빠름)에 풀고, **산출물은 드라이브**에 쓴다. 세션이 끊겨도
이어서 돌리기 위해서다 — 정렬은 한 세션에 안 끝날 수 있다.

In [ ]:
import os, zipfile, json
from google.colab import drive
drive.mount("/content/drive")

ZIP  = "/content/drive/MyDrive/vs01_align_upload.zip"   # ← 오디오+매니페스트 (1.75GiB)
SZIP = "/content/drive/MyDrive/vs01_align_scripts.zip"  # ← 스크립트 (22KB, 나중에 고친 것)
ROOT = "/content/608-root"        # audio-root. 압축 안 경로가 그대로 재현된다
OUT  = "/content/drive/MyDrive/vs01_segs"               # 산출물(드라이브 = 재개 가능)

os.makedirs(ROOT, exist_ok=True); os.makedirs(OUT, exist_ok=True)
if not os.path.exists(f"{ROOT}/align_manifest_vs01.jsonl"):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(ROOT)
# 스크립트는 **항상 덮어쓴다.** 처음 올린 zip에는 asr_eval/이 빠져 있어서
# 단어 분할이 조용히 꺼졌다(셀 3.5 참고). 최신 것이 이기게 둔다.
with zipfile.ZipFile(SZIP) as z:
    z.extractall(ROOT)
print(open(f"{ROOT}/VERSION.txt", encoding="utf-8").read())

wav = [p for d,_,fs in os.walk(ROOT) for p in fs if p.endswith(".wav")]
man = f"{ROOT}/align_manifest_vs01.jsonl"
rows = [json.loads(l) for l in open(man, encoding="utf-8")]
print(f"wav {len(wav)}개 · 매니페스트 {len(rows)}행")
assert len(wav) == 264 and len(rows) == 264, "264개가 아니다 — 업로드를 확인해라"

# 매니페스트가 가리키는 경로가 실제로 있는지 (align_608이 조용히 0개를 처리하는 걸 막는다)
miss = [r for r in rows if not os.path.exists(os.path.join(ROOT, r["expected_audio_relpath"]))]
print("경로 불일치:", len(miss))
assert not miss, miss[:2]

## 셀 3.5 — 검문: 단어 분할이 켜졌는가 (**건너뛰지 마라**)

2026-09-01에 여기서 70분을 버렸다. `align_608`은 단어/문장 판별을
`<자기 폴더>/asr_eval/baseline_asr.py`의 `task_type`으로 하는데, 그 파일을 안 올려서
임포트가 실패했다. **실패해도 죽지 않는다** — 경고 한 줄을 stderr에 찍고
`classify=None`이 되어 **전부 문장 모드**로 돈다.

그러면 단어 파일은 문장부호가 없으니 통짜 1문장이 되어 **파일당 세그먼트 1개**가
나온다. 264개를 다 돌고 나서야 `세그먼트 260개 / 부모 260개`로 드러났다.
그럴듯해 보이는 숫자라 더 위험하다.

→ 정렬을 시작하기 전에 여기서 막는다.

In [ ]:
import sys, os, importlib
# align_608과 **똑같은 방식**으로 임포트해 본다(경로 규칙이 바뀌면 같이 깨져야 한다)
sys.path.insert(0, os.path.join(ROOT, "asr_eval"))
for m in ("baseline_asr", "metrics", "split_speakers"):
    if m in sys.modules: importlib.reload(sys.modules[m])
try:
    from baseline_asr import task_type
except Exception as e:
    raise SystemExit(
        f"task_type 임포트 실패: {type(e).__name__}: {e}"
        + os.linesep + "→ vs01_align_scripts.zip을 드라이브에 올리고 셀 3을 다시 실행해라."
        + os.linesep + "   이대로 정렬하면 전부 문장 모드로 돌아 파일당 1세그먼트가 나온다.")

# 실제 전사로 판정이 되는지 본다
import json as _j
rows = [_j.loads(l) for l in open(f"{ROOT}/align_manifest_vs01.jsonl", encoding="utf-8")]
kinds = [task_type(r.get("transcript") or "") for r in rows]
n_word = kinds.count("wordlist")
print(f"task_type 판정: 단어 {n_word} · 문장 {kinds.count('narrative')} / {len(rows)}")
assert n_word == len(rows), "전부 wordlist여야 한다 — 매니페스트가 단어만 담고 있다"

# align_608의 이중 확인(_sentences==1)까지 통과하는지
sys.path.insert(0, ROOT)
from align_608 import _sentences
multi = [r["file_id"] for r in rows if len(_sentences(r.get("transcript") or "")) != 1]
print(f"_sentences가 1이 아닌 파일: {len(multi)}")
assert not multi, multi[:3]
print("→ 단어 분할이 켜진다. 정렬로 넘어가도 좋다.")

## 셀 3.6 — 산출물 치우기 (평소엔 **아무것도 안 한다**)

정렬을 처음부터 다시 해야 할 때만 쓴다. 예: 2026-09-01처럼 단어 분할이 꺼진 채로
돌아 파일당 1세그먼트가 나온 경우.

**왜 기본값이 꺼짐인가**: `align_608`은 `_attempted.txt`로 재개한다. 위에서부터
전부 실행하는 게 정상 흐름인데 이 셀이 기본으로 지우면 **매번 진행분이 날아간다.**
세션이 끊겨 다시 돌릴 때가 바로 그 상황이라 더 위험하다.

**지우지 않고 옆으로 치운다.** 되돌릴 수 있어야 한다 — 치운 폴더에 몇 시간이
들어 있을 수 있다.

In [ ]:
RESET_SEGS = False   # 처음부터 다시 정렬할 때만 True로. 평소엔 손대지 마라

import os, shutil, time, json
if not RESET_SEGS:
    if os.path.exists(f"{OUT}/segments.jsonl"):
        rows = [json.loads(l) for l in open(f"{OUT}/segments.jsonl", encoding="utf-8")]
        done = len({r["parent_file_id"] for r in rows})
        print(f"유지: 세그먼트 {len(rows)}개 · 부모 {done}/264 · 파일당 {len(rows)/max(done,1):.1f}개")
        print("(다시 정렬하려면 RESET_SEGS = True)")
    else:
        print("산출물이 아직 없다. 그대로 진행하면 된다.")
else:
    if os.path.exists(OUT):
        aside = f"{OUT}-폐기-{time.strftime('%m%d-%H%M')}"
        # 버리는 양을 숫자로 찍는다. "치웠다" 한 줄로는 대가가 안 보인다.
        lost = 0
        if os.path.exists(f"{OUT}/segments.jsonl"):
            lost = sum(1 for _ in open(f"{OUT}/segments.jsonl", encoding="utf-8"))
        shutil.move(OUT, aside)
        print(f"치웠다: {aside}  (세그먼트 {lost}개)")
        print("되돌리려면:")
        print(f"  !mv '{aside}' '{OUT}'")
    else:
        print("치울 것이 없다.")
    os.makedirs(OUT, exist_ok=True)
    print("빈 산출 폴더를 만들었다. 정렬 셀로 넘어가라.")

## 셀 4 — 정렬 (오래 걸린다 · 끊기면 이 셀만 다시)

`align_608.py`는 `_attempted.txt`로 재개한다. 산출물이 드라이브에 있으므로
세션이 끊겨도 **셀 1~3을 다시 돌리고 이 셀을 재실행**하면 이어서 간다.

- `--model small` : 이전 5개 배치와 동일. 바꾸지 마라
- `--wordlist auto` : 전사 문장부호로 단어/문장 자동 판별
- 예상: T4에서 수 시간 (로컬 CPU 29시간 대비)

In [ ]:
import time, subprocess, os
t0 = time.time()
cmd = ["python", f"{ROOT}/align_608.py",
       "--manifest", f"{ROOT}/align_manifest_vs01.jsonl",
       "--audio-root", ROOT,
       "--out-dir", OUT,
       "--model", "small",
       "--wordlist", "auto"]
print(" ".join(cmd), flush=True)
subprocess.run(cmd, check=False)      # 실패해도 아래 진행 상황은 보고 싶다
print()
print(f"경과 {(time.time()-t0)/60:.1f}분")

seg = f"{OUT}/segments.jsonl"
if os.path.exists(seg):
    rows = [json.loads(l) for l in open(seg, encoding="utf-8")]
    done = len({r["parent_file_id"] for r in rows})
    per = len(rows) / max(done, 1)
    print(f"세그먼트 {len(rows)}개 · 처리한 부모 {done}/264 · 파일당 {per:.1f}개")
    # TS01은 단어 70파일에서 1515세그(파일당 21.6)가 나왔다. 파일당 1~2개면
    # 단어 분할이 꺼진 것이다(셀 3.5 참고) — 그대로 두면 70분을 또 버린다.
    if per < 5:
        raise SystemExit(
            f"파일당 세그먼트가 {per:.1f}개뿐이다 — 문장 모드로 돌았을 가능성이 크다."
            + os.linesep + "셀 3.5를 실행해 원인을 확인하고, OUT 폴더를 치운 뒤 다시 정렬해라.")
    if done < 264:
        print("→ 아직 남았다. 이 셀을 다시 실행하면 이어서 간다.")

## 셀 5 — 재정렬 (필수)

`align_608`의 경계는 whisper 타임스탬프라 앞뒤 침묵이 많고 끝이 잘리기도 한다.
대장 3절·3-2절이 그 실측이다 — **이 단계를 건너뛰면 학습이 망가진다.**

단어만 있으므로 `--narrative skip`(기본)이면 된다.

In [ ]:
import subprocess, json, collections, time
REF = OUT + "_refined"
t0 = time.time()
subprocess.run(["python", f"{ROOT}/refine_segments.py",
                "--segments", f"{OUT}/segments.jsonl",
                "--audio-root", ROOT,
                "--out-dir", REF,
                "--resume"], check=False)
print(f"경과 {(time.time()-t0)/60:.1f}분")

rows = [json.loads(l) for l in open(f"{REF}/segments.jsonl", encoding="utf-8")]
c = collections.Counter(r.get("task_type") for r in rows)
dur = sum(r["end"] - r["start"] for r in rows)
print(f"재정렬 후 {len(rows)}개 · {dict(c)} · 총 {dur/60:.1f}분")
print("단어 세그가 대부분이어야 정상 (TS01 70파일 → 1515세그 기준이면 264파일은 약 5700개)")

## 셀 6 — 회수

로컬로 가져가 `prepare_colab_trainset.py`에 넣는다. 세그먼트 wav는 0.6초짜리라
전부 합쳐도 작다.

In [ ]:
import shutil, os
base = "/content/drive/MyDrive/vs01_segs_refined"
out = shutil.make_archive(base, "zip", REF)
print("회수용 zip:", out, f"{os.path.getsize(out)/2**20:.0f} MB")
print()
print("로컬에서 할 일:")
print("  1. 이 zip을 ~/Downloads/608-audio-work/_segs_vs01_refined/ 로 풀기")
print("  2. _segs_merged_all6.jsonl 과 병합 → _segs_merged_all7.jsonl")
print("  3. prepare_colab_trainset.py --test-speaker-frac 0.2 --dev-speaker-frac 0.1")
print("     --min-sec 0.3 --max-sec 30  (min-sec을 빼면 단어가 전멸한다)")